In [ ]:
# Enter API key 
import os
from getpass import getpass

openai_api_key = getpass("Enter your OpenAI API Key: ")
os.environ["OPENAI_API_KEY"] = openai_api_key


if openai_api_key and len(openai_api_key) > 20:
    print("API Key entered successfully!")
else:
    print("The API Key you entered is invalid. Please try again.")

API Key entered successfully!


In [ ]:
# Load all PDFs from documents folder
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

loader = DirectoryLoader(
    "documents/",
    glob="**/*.pdf",
    loader_cls=PyPDFLoader,
    show_progress=True  # Shows loading progress
)

documents = loader.load()

print(f" Successfully loaded {len(documents)} pages total")
print(f"\n From {len([f for f in os.listdir('documents') if f.endswith('.pdf')])} PDF files")

# Show a preview of the first page
print(f"\n First page preview (first 400 characters):")
print(documents[0].page_content[:400])
print(f"\n Source: {documents[0].metadata['source']}")
print(f" Page: {documents[0].metadata.get('page', 'N/A')}")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
100%|██████████| 20/20 [00:24<00:00,  1.23s/it]

 Successfully loaded 478 pages total

 From 20 PDF files

 First page preview (first 400 characters):
Predicting cancer outcomes from histology and genomics using convolutional networks 
Author(s): Pooya Mobadersany, Safoora Yousefi, Mohamed Amgad, David A. Gutman, Jill 
S. Barnholtz-Sloan, José E. Velázquez Vega, Daniel J. Brat and Lee A. D. Cooper  
Source: Proceedings of the National Academy of Sciences of the United States of 
America , March 27, 2018, Vol. 115, No. 13 (March 27, 2018), pp. E2

 Source: documents/Predicting cancer outcomes from histology and genomics using convolutional networks.pdf
 Page: 0


In [ ]:
# Chunking the documents into smaller pieces
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,        
    chunk_overlap=200,      
    length_function=len,    
    separators=["\n\n", "\n", " ", ""])

# Split all documents into chunks
chunks = text_splitter.split_documents(documents)

print(f" Split {len(documents)} pages into {len(chunks)} chunks")
print(f"\n Original pages: {len(documents)}")
print(f" New chunks: {len(chunks)}")
print(f" Average chunks per page: {len(chunks)/len(documents):.1f}")


print(f"\n  Example chunk:")
print(f"Length: {len(chunks[0].page_content)} characters")
print(f"\nContent preview:")
print(chunks[0].page_content[:300])
print(f"\nMetadata: {chunks[0].metadata}")

 Split 478 pages into 2184 chunks

 Original pages: 478
 New chunks: 2184
 Average chunks per page: 4.6

  Example chunk:
Length: 983 characters

Content preview:
Predicting cancer outcomes from histology and genomics using convolutional networks 
Author(s): Pooya Mobadersany, Safoora Yousefi, Mohamed Amgad, David A. Gutman, Jill 
S. Barnholtz-Sloan, José E. Velázquez Vega, Daniel J. Brat and Lee A. D. Cooper  
Source: Proceedings of the National Academy of S

Metadata: {'producer': 'Acrobat Distiller 10.0.0 (Windows); modified using iText® 7.1.3 ©2000-2018 iText Group NV (JSTOR Michigan; licensed version)', 'creator': 'Arbortext Advanced Print Publisher 9.1.510/W Unicode', 'creationdate': '2018-03-17T07:43:33+05:30', 'crossmarkdomains[1]': 'www.pnas.org', 'crossmarkdomainexclusive': 'false', 'crossmarkmajorversiondate': '2018-03-17', 'moddate': '2020-12-09T06:24:21+00:00', 'title': 'Predicting cancer outcomes from histology and genomics using convolutional networks', 'doi': '10.1073/p

In [ ]:
# Create embeddings and store in vector database
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

# Initialize OpenAI embeddings
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"  # OpenAI's embedding model
)

print(" Creating embeddings forchunks...")
print("  This will take 1-2 minutes and will use your OpenAI API credits")
print("   (Approximately $0.001-0.002 for this dataset)\n")


# Create vector store from documents
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"  # Save to disk 
)


print(f" Created vector database with {len(chunks)} embedded chunks!")
print(f" Database saved to './chroma_db' folder")
print(f"\n You can now search through your documents!")

 Creating embeddings forchunks...
  This will take 1-2 minutes and will use your OpenAI API credits
   (Approximately $0.001-0.002 for this dataset)

 Created vector database with 2184 embedded chunks!
 Database saved to './chroma_db' folder

 You can now search through your documents!


In [ ]:
# Test similarity search
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings
)

# Test query
test_query = "What are the challenges in brain tumor diagnosis?"

print(f" Searching for: '{test_query}'\n")

# Find the top 3 most relevant chunks
results = vectorstore.similarity_search(test_query, k=3)

for i, doc in enumerate(results, 1):
    print(f"{'='*60}")
    print(f" Result {i}")
    print(f"{'='*60}")
    print(f"Source: {doc.metadata['source']}")
    print(f"Page: {doc.metadata.get('page', 'N/A')}")
    print(f"\nContent:")
    print(doc.page_content[:500])  # First 400 characters
    print(f"\n")

/var/folders/0q/2fgz4p297432bjytntcz1lfh0000gn/T/ipykernel_39629/1335786048.py:8: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


 Searching for: 'What are the challenges in brain tumor diagnosis?'

 Result 1
Source: documents/Brain Tumor Detection Based on Deep Learning Approaches and Magnetic Resonance Imaging.pdf
Page: 1

Content:
ventional biopsy techniques are painful, time-consuming, and fraught with inaccuracy in
sampling [13,14]. Histopathological tumor grading (Biopsy) has its own set of problems,
including intra-tumor heterogeneity and differences in the subjective assessments of differ-
ent experts [15]. The diagnostic process for tumors is made more difﬁcult and restrictive by
these characteristics.
Effective treatment planning and patient outcomes depend on a quick and precise
diagnosis of brain tumors. However,


 Result 2
Source: documents/Deep Learning Approaches for Brain Tumor Detection and Classification Using MRI Images.pdf
Page: 0

Content:
differentiation and segmentation become exceptionally dif-
ﬁcult. Broadly, brain tumors are categorized as either benign
or malignant. Benign tumors, whic

In [ ]:
# Build RAG system manually
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

template = """You are a helpful research assistant specializing in medical AI and brain tumor detection.
Use the following pieces of context to answer the question at the end. 

If you don't know the answer based on the context provided, just say "I don't have enough information in the provided documents to answer that question." Don't make up an answer.

If the answer is in the context, provide a detailed answer and mention which study or paper it comes from if possible.

Context:
{context}

Question: {question}

Detailed Answer:"""

prompt = ChatPromptTemplate.from_template(template)


retriever = vectorstore.as_retriever(search_kwargs={"k": 4}) # Retrieve top 4 chunks

print(" RAG system ready!")
print(" Using GPT-4o-mini for answers")
print(" Retrieving top 4 most relevant chunks per question")

 RAG system ready!
 Using GPT-4o-mini for answers
 Retrieving top 4 most relevant chunks per question


In [ ]:
# Ask a question (manual approach)
def format_docs(docs):
    """Combine retrieved documents into a single string"""
    return "\n\n".join(doc.page_content for doc in docs)

question = "What deep learning architectures are used for brain tumor detection in these papers?"
print(f" Question: {question}\n")
print(" Processing...\n")

source_docs = retriever.invoke(question)
context = format_docs(source_docs)

messages = prompt.format_messages(context=context, question=question)
answer = llm.invoke(messages)

print("="*70)
print(" ANSWER:")
print("="*70)
print(answer.content)

print("\n" + "="*70)
print(" SOURCES USED:")
print("="*70)
for i, doc in enumerate(source_docs, 1):
    print(f"\n{i}. {doc.metadata['source']} (Page {doc.metadata.get('page', 'N/A')})")
    print(f"   Preview: {doc.page_content[:150]}...")

 Question: What deep learning architectures are used for brain tumor detection in these papers?

 Processing...

 ANSWER:
The survey study by K. M. Hosny and M. A. Mohammed reviews various deep learning architectures used for brain tumor detection. The architectures highlighted in the context include:

1. **Convolutional Neural Networks (CNNs)**: These are the primary deep learning models discussed, known for their hierarchical approach to feature extraction. CNNs utilize convolutional layers, pooling layers, and fully connected layers to capture features such as edges and textures in MRI images.

2. **Transfer Learning**: This approach involves using pre-trained models on new datasets, which can enhance performance, especially when the available data is limited.

3. **Vision Transformers (ViTs)**: A newer architecture that applies transformer models to image data, which has shown promise in various image classification tasks, including medical imaging.

4. **Hybrid Techniques**: These

In [ ]:
# Interactive Q&A Session
print("="*70)
print("BRAIN TUMOR DETECTION RAG RESEARCH ASSISTANT")
print("="*70)
print("\n Loaded documents ready to answer your questions!")
print("Press Ctrl+C to force stop\n")
print("="*70)

def ask_question(question):
    """
    Takes a question, retrieves relevant docs, adn generates an answer.

    This function encapsulates the RAG pipeline:
    1. Retrieve relevant chunks from vector store
    2. Format them as context
    3. Send to LLM with prompt
    4. Return answer and sources
    """

    # Step 1: Retrieve relevant documents (Top 4 Chunks)
    source_docs = retriever.invoke(question)

    # Step 2: Combine retrieved docs into single context string
    context = format_docs(source_docs)

    # Step 3: Create the prompt with context and question
    messages = prompt.format_messages(context=context, question=question)

    # Step 4: Get answer from LLM
    answer = llm.invoke(messages)

    return answer.content, source_docs

# Main interaction loop
while True:
    user_question = input("\nYour Question (or 'quit' to exit):").strip()

    if user_question.lower() in ['quit', 'exit', 'q']:
        print("\nThanks for using the Research Assistant! Goodbye!")
        break

    if not user_question:
        print("Please enter a question!")
        continue

    print("\nSearching documents and generating answer...\n")


    try:
        answer, sources = ask_question(user_question)

        print("="*70)
        print(" Answer:")
        print("="*70)
        print(answer)

        print("\n" + "="*70)
        print("SOURCES USED:")
        print("="*70)
        for i, doc in enumerate(sources, 1):
            # Extract filename from full path
            filename = doc.metadata['source'].split('/')[-1]
            page_num = doc.metadata.get('page', 'N/A')
            print(f"\n{i}. {filename}")
            print(f" Page: {page_num}")
            print(f" Preview: {doc.page_content[120]}...")
        
        print("\n" + "="*70)

    except Exception as e:
        print("fError: {e}")
        print("Please try asking your question differently.")

BRAIN TUMOR DETECTION RAG RESEARCH ASSISTANT

 Loaded documents ready to answer your questions!
Press Ctrl+C to force stop


Searching documents and generating answer...

 Answer:
Convolutional Neural Networks (CNNs) are a class of deep learning algorithms specifically designed for processing structured grid data, such as images. They are particularly effective in tasks involving image recognition and classification due to their ability to automatically learn spatial hierarchies of features from the input data. CNNs consist of multiple layers, including convolutional layers, pooling layers, and fully connected layers, which work together to extract features and make predictions.

In the context of tumor detection, CNNs are utilized in several ways:

1. **Image Classification**: CNNs can classify MRI images into categories such as healthy or tumor-present. For instance, a two-channel CNN was used to initially classify MRI images into healthy and tumor categories, and if a tumor is detec

In [ ]:
# Enhanced RAG System with Conversation History
from langchain_core.messages import HumanMessage, AIMessage

print("="*70)
print("BRAIN TUMOR DETECTION RAG RESEARCH ASSISTANT (WITH MEMORY)")
print("="*70)
print("\nLoaded documents ready to answer your questions!")
print("This system remembers your conversation for context.")
print("Type 'quit', 'exit', or 'q' to end the session")
print("Type 'history' to see your conversation history")
print("Press Ctrl+C to force stop\n")
print("="*70)

# Initialize conversation history
conversation_history = []

def ask_question_with_memory(question):
    """
    Takes a question, retrieves relevant docs, and generates an answer.
    Now includes conversation history for context.
    
    Parameters:
    - question: The user's question
    
    Returns:
    - answer: The AI's response
    - source_docs: The retrieved documents used
    """

    # Retrieve relevant documents (top 4 chunks)
    source_docs = retriever.invoke(question)
    context = format_docs(source_docs)
    
    # Build conversation context from history
    history_context = ""
    if conversation_history:
        recent_history = conversation_history[-3:]  # Last 3 exchanges
        history_context = "\n\nPrevious conversation:\n"
        for i, exchange in enumerate(recent_history, 1):
            history_context += f"Q{i}: {exchange['question']}\n"
            history_context += f"A{i}: {exchange['answer']}\n\n"
    
    full_context = context + history_context
    messages = prompt.format_messages(context=full_context, question=question)
    
    answer = llm.invoke(messages)
    

    conversation_history.append({
        'question': question,
        'answer': answer.content
    })
    
    return answer.content, source_docs

def display_history():
    """Display the conversation history"""
    if not conversation_history:
        print("\nNo conversation history yet.")
        return
    
    print("\n" + "="*70)
    print("CONVERSATION HISTORY")
    print("="*70)
    for i, exchange in enumerate(conversation_history, 1):
        print(f"\nQ{i}: {exchange['question']}")
        print(f"A{i}: {exchange['answer'][:200]}...")  # First 200 chars
    print("="*70)

question_count = 0

while True:
    user_question = input(f"\n[Q{question_count + 1}] Your Question (or 'quit' to exit): ").strip()

    if user_question.lower() in ['quit', 'exit', 'q']:
        print(f"\nSession Summary: {question_count} questions answered")
        print("Thanks for using the Research Assistant! Goodbye!")
        break
    
    if user_question.lower() == 'history':
        display_history()
        continue

    if not user_question:
        print("Please enter a question!")
        continue
    
    question_count += 1
    print("\nSearching documents and generating answer...\n")
    
    try:
        answer, sources = ask_question_with_memory(user_question)

        print("="*70)
        print(f"ANSWER (Question {question_count}):")
        print("="*70)
        print(answer)
        
        print("\n" + "="*70)
        print("SOURCES USED:")
        print("="*70)
        for i, doc in enumerate(sources, 1):
            # Extracts filename from full path
            filename = doc.metadata['source'].split('/')[-1]
            page_num = doc.metadata.get('page', 'N/A')
            print(f"\n{i}. {filename}")
            print(f"   Page: {page_num}")
            print(f"   Preview: {doc.page_content[:120]}...")
        
        print("\n" + "="*70)
        
    except Exception as e:
        print(f"Error: {e}")
        print("Please try asking your question differently.")

BRAIN TUMOR DETECTION RAG RESEARCH ASSISTANT (WITH MEMORY)

Loaded documents ready to answer your questions!
This system remembers your conversation for context.
Type 'quit', 'exit', or 'q' to end the session
Type 'history' to see your conversation history
Press Ctrl+C to force stop


Searching documents and generating answer...

ANSWER (Question 1):
Commonly used machine learning architectures for brain tumor detection include:

1. **Support Vector Machines (SVM)**: SVM is a widely utilized algorithm for classification and regression tasks, including medical image analysis. It works by identifying an optimal hyperplane that maximizes the margin between different classes, which is particularly effective for tumor classification.

2. **Convolutional Neural Networks (CNNs)**: Various CNN architectures have been employed for brain tumor detection. Notable examples include:
   - **GoogLeNet**
   - **InceptionV3**
   - **DenseNet201**
   - **AlexNet**
   - **ResNet50**

   These architectur